# Notebook 10 — Uncertainty-Aware Terrain Risk / Cost Map & Offline A* Path Planning
**Project**: Uncertainty-Aware Terrain Semantic Segmentation and Traversability Estimation for Autonomous Planetary Rover Navigation  
**Stage B**: Incorporating Predictive Uncertainty into 2D Cost Maps for Offline Image-Space Rover Route Planning

---

## 1. Research Objectives & Methodology
This notebook investigates:
> *"Can predictive uncertainty be incorporated into a terrain risk/cost map so that offline path planning prefers lower-risk regions?"*

We perform a strict paired comparison between:
- **Method A (Baseline Semantic Cost)**:
  $$C_A(x, y) = C_s(x, y)$$
- **Method B (Uncertainty-Aware Cost)**:
  $$C_u(x, y) = \text{round}(255 \cdot H_{\text{norm}}(p(x, y)))$$
  $$C_B(x, y) = \begin{cases} 255 & \text{if } C_s(x, y) \ge 240 \\ \min(239, \text{round}(0.7 \cdot C_s(x, y) + 0.3 \cdot C_u(x, y))) & \text{otherwise} \end{cases}$$

All evaluation is conducted on a fixed $256 \times 256$ pixel grid with identical start `(5, 5)` and goal `(250, 250)` coordinates on the 190 validation frames.  
**Note**: This evaluates simulated, offline image-space path planning, not physical rover navigation.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else (next((p for p in (cwd, *cwd.parents) if (p / 'notebooks').is_dir()), cwd))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "risk_aware_navigation"

# Load aggregate summary
with open(OUTPUT_DIR / "risk_map_summary.json", "r") as f:
    summary = json.load(f)

print("=== Stage B Path Planning Aggregate Summary ===")
print(f"Validation Images Evaluated: {summary['evaluation_protocol']['images_evaluated']}")
print(f"Method A (Semantic Baseline)   - Success Rate: {summary['method_a_baseline']['success_rate_pct']:.1f}% ({summary['method_a_baseline']['successful_paths']}/{summary['evaluation_protocol']['images_evaluated']})")
print(f"Method B (Uncertainty-Aware)   - Success Rate: {summary['method_b_uncertainty_aware']['success_rate_pct']:.1f}% ({summary['method_b_uncertainty_aware']['successful_paths']}/{summary['evaluation_protocol']['images_evaluated']})")
print()
print("=== Paired Succeeded Comparison ===")
paired = summary['paired_comparison']
print(f"Both Succeeded:                {paired['both_succeeded']} frames")
print(f"Number of Changed Paths:       {paired['paths_changed_count']} frames ({paired['paths_changed_pct']:.1f}%)")
print(f"Paths where B has Lower Uncert:{paired['b_lower_uncertainty_count']} frames ({paired['b_lower_uncertainty_pct']:.1f}%)")
print(f"Mean Path Uncertainty A:       {paired['mean_path_uncertainty_A']:.4f}")
print(f"Mean Path Uncertainty B:       {paired['mean_path_uncertainty_B']:.4f} (Reduction: {paired['mean_uncertainty_reduction_pct']:.1f}%)")
print(f"Mean Path Length A:            {paired['mean_path_length_A']:.2f} px")
print(f"Mean Path Length B:            {paired['mean_path_length_B']:.2f} px (Length Overhead: {paired['mean_length_overhead_pct']:.1f}%)")

## 2. Paired Navigation Evaluation Table
Detailed breakdown of each validation frame comparing path length, path cost, and mean accumulated uncertainty.

In [ ]:
df_nav = pd.read_csv(OUTPUT_DIR / "baseline_vs_uncertainty_metrics.csv")
display_cols = ['image_id', 'a_found', 'b_found', 'path_changed', 'a_length', 'b_length', 'a_mean_u', 'b_mean_u', 'u_diff']
print(df_nav[display_cols].head(15).to_markdown(index=False))

## 3. Representative Route Visualizations
Comparing Method A (Cyan) and Method B (Orange) routes plotted over the RGB image, Ground-Truth Mask, Predicted Semantics, Cost Maps, and Predictive Entropy.

In [ ]:
vis_paths = sorted((OUTPUT_DIR / "representative_paths").glob("*.png"))
for vp in vis_paths:
    print(f"Visualizing: {vp.name}")
    img = Image.open(vp)
    plt.figure(figsize=(18, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()